<a href="https://colab.research.google.com/github/office268/ipynb/blob/main/langgraph_tavily_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# סוכן AI עם LangGraph + Tavily + Gemini

סוכן פשוט שמחפש מידע באינטרנט באמצעות Tavily ועונה בעזרת מודל Gemini של Google.

In [ ]:
!pip install -q langgraph langchain-google-genai langchain-community tavily-python

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
except Exception:
    os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
    os.environ["TAVILY_API_KEY"] = "YOUR_TAVILY_API_KEY"

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults

# כלי חיפוש
tools = [TavilySearchResults(max_results=3)]

# מודל Gemini עם גישה לכלי החיפוש
model = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"]
).bind_tools(tools)

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

# צומת המודל
def call_model(state: MessagesState):
    return {"messages": [model.invoke(state["messages"])]}

# בדיקה: האם המודל ביקש להפעיל כלי?
def should_continue(state: MessagesState):
    return "tools" if state["messages"][-1].tool_calls else END

# בניית הגרף
graph = StateGraph(MessagesState)
graph.add_node("model", call_model)
graph.add_node("tools", ToolNode(tools))

graph.add_edge(START, "model")
graph.add_conditional_edges("model", should_continue)
graph.add_edge("tools", "model")

agent = graph.compile()

# הצגת הגרף
try:
    from IPython.display import Image, display
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    print("START → model → tools → model → ... → END")

In [ ]:
from langchain_core.messages import HumanMessage

# שאל את הסוכן שאלה
question = "מה החדשות הכי מעניינות בתחום ה-AI היום?"

result = agent.invoke({"messages": [HumanMessage(content=question)]})

# הדפסת התשובה הסופית
print(result["messages"][-1].content)